# LAB 04 — Experiment 3: Representation comparison
So sánh các biểu diễn văn bản trên IMDB với cùng split phân tầng train/validation/test = 60/20/20.

- Model A: TF-IDF + Logistic Regression
- Model B: Average Word2Vec + Logistic Regression
- Model C (thêm): TF-IDF + Linear SVM

Bảng cuối in training time, vocabulary/embedding vocabulary, accuracy, precision, recall, F1 trên test. Word2Vec chỉ được huấn luyện bằng review trong train để tránh leakage.

In [4]:
# Gensim dùng để huấn luyện Word2Vec. Nếu Colab chưa có, cài package.
try:
    import gensim
except ImportError:
    %pip -q install gensim

# Tương thích với một số phiên bản SciPy mới đã bỏ scipy.linalg.triu
import scipy.linalg, numpy as np
if not hasattr(scipy.linalg, 'triu'):
    scipy.linalg.triu = np.triu
from gensim.models import Word2Vec
print('gensim', gensim.__version__)

gensim 4.4.0


In [5]:
from pathlib import Path
import zipfile, time, re, html
import pandas as pd

ZIP_PATH = Path('/content/imdb.zip')

if not ZIP_PATH.exists():
    from google.colab import files
    print('Chọn imdb.zip để tải lên Colab.')
    uploaded = files.upload()
    ZIP_PATH = Path(next(iter(uploaded)))

def load_imdb_zip(zip_path):
    frames=[]
    with zipfile.ZipFile(zip_path) as z:
        csvs=[n for n in z.namelist() if n.lower().endswith('.csv')]
        if not csvs:
            raise ValueError('Không tìm thấy CSV trong ZIP.')
        for name in csvs:
            with z.open(name) as f: part=pd.read_csv(f)
            part.columns=[str(c).strip().lower() for c in part.columns]
            if {'review','sentiment'}.issubset(part.columns):
                part=part[['review','sentiment']].copy()
            elif {'text','label'}.issubset(part.columns):
                part=part.rename(columns={'text':'review','label':'sentiment'})[['review','sentiment']].copy()
            else:
                raise ValueError(f'{name}: cần cột review/sentiment hoặc text/label; thấy {list(part.columns)}')
            frames.append(part)
    return pd.concat(frames,ignore_index=True)

df=load_imdb_zip(ZIP_PATH)
df['review']=df['review'].fillna('').astype(str)
df['sentiment']=df['sentiment'].astype(str).str.strip().str.lower()
df=df[df['sentiment'].isin(['positive','negative'])].copy()
df=df.drop_duplicates(subset='review').reset_index(drop=True)
df['label']=df['sentiment'].map({'negative':0,'positive':1})
print('Số review sau loại trùng:', f'{len(df):,}')
display(df['sentiment'].value_counts().to_frame('count'))

def normalize_review(text):
    text=html.unescape(text)
    text=re.sub(r'<[^>]+>', ' ', text)
    text=re.sub(r'\s+', ' ', text).strip().lower()
    return text
df['text_clean']=df['review'].map(normalize_review)

Chọn imdb.zip để tải lên Colab.


Saving imdb.zip to imdb.zip
Số review sau loại trùng: 49,582


,count
sentiment,
positive,24884
negative,24698


## Chia dữ liệu 60/20/20
Dùng random_state=42 và stratify=y, cùng cách chia với Experiment 2. Review trùng được loại trước khi chia. Test chỉ dùng ở bước đánh giá cuối.

In [6]:
from sklearn.model_selection import train_test_split
X=df['text_clean']; y=df['label']
X_trainval,X_test,y_trainval,y_test=train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)
X_train,X_val,y_train,y_val=train_test_split(X_trainval,y_trainval,test_size=0.25,random_state=42,stratify=y_trainval)
print(f'Train={len(X_train):,}; Validation={len(X_val):,}; Test={len(X_test):,}')

Train=29,748; Validation=9,917; Test=9,917


## Model A — TF-IDF + Logistic Regression
TF-IDF vocabulary/IDF được fit trên train bên trong Pipeline.

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

def get_metrics(name, representation, classifier, y_true, y_pred, seconds, vocab_size):
    return {'Model':name,'Representation':representation,'Classifier':classifier,
            'Training time (s)':seconds,'Vocabulary size':vocab_size,
            'Accuracy':accuracy_score(y_true,y_pred),
            'Precision':precision_score(y_true,y_pred,zero_division=0),
            'Recall':recall_score(y_true,y_pred,zero_division=0),
            'F1':f1_score(y_true,y_pred,zero_division=0)}

results=[]
tfidf_lr=Pipeline([
    ('tfidf',TfidfVectorizer(max_features=50000,ngram_range=(1,1),min_df=2,
                             strip_accents='unicode',sublinear_tf=True)),
    ('classifier',LogisticRegression(max_iter=1000,random_state=42))
])
t0=time.perf_counter(); tfidf_lr.fit(X_train,y_train); tfidf_lr_seconds=time.perf_counter()-t0
y_pred_a=tfidf_lr.predict(X_test)
tfidf_vocab=len(tfidf_lr.named_steps['tfidf'].vocabulary_)
results.append(get_metrics('Model A','TF-IDF','Logistic Regression',y_test,y_pred_a,tfidf_lr_seconds,tfidf_vocab))
print('Model A test confusion matrix [[TN, FP], [FN, TP]]:')
print(confusion_matrix(y_test,y_pred_a,labels=[0,1]))
print(f'TF-IDF vocabulary size: {tfidf_vocab:,}')
print(f'Train time: {tfidf_lr_seconds:.2f} seconds')

Model A test confusion matrix [[TN, FP], [FN, TP]]:
[[4361  579]
 [ 445 4532]]
TF-IDF vocabulary size: 48,484
Train time: 8.01 seconds


## Model B — Average Word2Vec + Logistic Regression
Huấn luyện Word2Vec trên token của train; biểu diễn mỗi review bằng trung bình vector của các từ có trong vocabulary. Review không có từ nào trong vocabulary nhận vector 0. Với classifier, dùng cùng Logistic Regression.

In [8]:
def tokenize(text):
    return re.findall(r'\b\w+\b',text.lower())

train_tokens=[tokenize(t) for t in X_train]
w2v_size=100
t0=time.perf_counter()
w2v=Word2Vec(
    sentences=train_tokens, vector_size=w2v_size, window=5, min_count=2,
    workers=2, sg=1, epochs=5, seed=42
)
def average_vector(text, model, dim):
    vectors=[model.wv[w] for w in tokenize(text) if w in model.wv]
    return np.mean(vectors,axis=0) if vectors else np.zeros(dim,dtype=np.float32)

X_train_w2v=np.vstack([average_vector(t,w2v,w2v_size) for t in X_train])
X_test_w2v=np.vstack([average_vector(t,w2v,w2v_size) for t in X_test])
lr_w2v=LogisticRegression(max_iter=1000,random_state=42)
lr_w2v.fit(X_train_w2v,y_train)
w2v_seconds=time.perf_counter()-t0
y_pred_b=lr_w2v.predict(X_test_w2v)
results.append(get_metrics('Model B','Average Word2Vec','Logistic Regression',y_test,y_pred_b,w2v_seconds,len(w2v.wv)))
print('Model B test confusion matrix [[TN, FP], [FN, TP]]:')
print(confusion_matrix(y_test,y_pred_b,labels=[0,1]))
print(f'Word2Vec vocabulary size: {len(w2v.wv):,}; vector size: {w2v_size}')
print(f'Train time (Word2Vec + Logistic Regression): {w2v_seconds:.2f} seconds')

Model B test confusion matrix [[TN, FP], [FN, TP]]:
[[4230  710]
 [ 661 4316]]
Word2Vec vocabulary size: 51,395; vector size: 100
Train time (Word2Vec + Logistic Regression): 253.19 seconds


## Model C (tùy chọn) — TF-IDF + Linear SVM
Cùng biểu diễn TF-IDF và split, thay classifier bằng LinearSVC để có thêm một baseline tham chiếu.

In [9]:
tfidf_svm=Pipeline([
    ('tfidf',TfidfVectorizer(max_features=50000,ngram_range=(1,1),min_df=2,
                             strip_accents='unicode',sublinear_tf=True)),
    ('classifier',LinearSVC(random_state=42))
])
t0=time.perf_counter(); tfidf_svm.fit(X_train,y_train); svm_seconds=time.perf_counter()-t0
y_pred_c=tfidf_svm.predict(X_test)
svm_vocab=len(tfidf_svm.named_steps['tfidf'].vocabulary_)
results.append(get_metrics('Model C','TF-IDF','Linear SVM',y_test,y_pred_c,svm_seconds,svm_vocab))
print('Model C test confusion matrix [[TN, FP], [FN, TP]]:')
print(confusion_matrix(y_test,y_pred_c,labels=[0,1]))
print(f'Train time: {svm_seconds:.2f} seconds')

Model C test confusion matrix [[TN, FP], [FN, TP]]:
[[4376  564]
 [ 501 4476]]
Train time: 8.06 seconds


## Bảng so sánh và trả lời
Metrics được tính trên cùng test set; positive là lớp dương. Thời gian là fit representation + classifier trên train, không gồm thời gian khởi động Colab hay dự đoán test.

In [10]:
results_df=pd.DataFrame(results)
display(results_df.style.format({
    'Training time (s)':'{:.2f}','Accuracy':'{:.4f}',
    'Precision':'{:.4f}','Recall':'{:.4f}','F1':'{:.4f}'
}))

best_row=results_df.loc[results_df['F1'].idxmax()]
print(f"F1 cao nhat tren test: {best_row['Model']} ({best_row['F1']:.4f}).")
print('Doi chieu them Accuracy, Precision, Recall, training time va confusion matrix; khong ket luan chi tu accuracy.')


,Model,Representation,Classifier,Training time (s),Vocabulary size,Accuracy,Precision,Recall,F1
0,Model A,TF-IDF,Logistic Regression,8.01,48484,0.8967,0.8867,0.9106,0.8985
1,Model B,Average Word2Vec,Logistic Regression,253.19,51395,0.8618,0.8587,0.8672,0.8629
2,Model C,TF-IDF,Linear SVM,8.06,48484,0.8926,0.8881,0.8993,0.8937


F1 cao nhat tren test: Model A (0.8985).
Doi chieu them Accuracy, Precision, Recall, training time va confusion matrix; khong ket luan chi tu accuracy.


### Trả lời câu hỏi: Tại sao hai representation tạo ra kết quả khác nhau?
TF-IDF tạo đặc trưng thưa theo từ/cụm từ và giữ tín hiệu từ vựng trực tiếp; Logistic Regression có thể học các từ gắn với positive/negative. Average Word2Vec tạo vector dày theo ngữ nghĩa nhưng phép lấy trung bình làm mất thứ tự từ và có thể làm giảm vai trò của từ quan trọng như từ phủ định. Vì vậy hai cách biểu diễn có thể cho kết quả khác nhau. Hãy dựa vào bảng test và confusion matrix để nhận xét dựa trên kết quả chạy thực tế.

Lưu ý: nếu muốn phép so sánh chỉ tập trung vào representation, so sánh Model A và B vì cùng dùng Logistic Regression. Model C thay cả classifier nên là baseline bổ sung, không phải so sánh representation thuần túy.